# ⚡ VisionProf — Google Colab Experiment Runner

Notebook này phục vụ chạy toàn bộ thực nghiệm chuẩn hóa của **VisionProf** trên **Google Colab (CPU & T4 GPU)**,
đồng bộ dữ liệu kết quả về Google Drive và mở cổng Live Dashboard qua tunnel **ngrok**.

In [ ]:
# 1. Kiểm tra phần cứng
!nvidia-smi
!lscpu || cat /proc/cpuinfo | grep 'model name' | head -n 2

In [ ]:
# 2. Cài đặt thư viện phụ thuộc và pyngrok
!pip install -r requirements.txt -q
!pip install pyngrok -q
!python scripts/verify_rules.py

In [ ]:
# 3. Chạy thực nghiệm mẫu (TN3: Baseline 8 models trên Colab T4)
!python scripts/run_experiment.py --model resnet50 --device colab_t4 --batch-size 1 --precision fp32 --experiment tn3
!python scripts/run_experiment.py --model bert_base --device colab_t4 --batch-size 1 --precision fp32 --experiment tn3
!python scripts/run_experiment.py --model xgboost --device colab_t4 --batch-size 1 --precision fp32 --experiment tn3

In [ ]:
# 4. Sao lưu dữ liệu results/ vào Google Drive
from google.colab import drive
drive.mount('/content/drive')
!tar -czf /content/drive/MyDrive/visionprof_results.tar.gz results/
print("✅ Đã sao lưu results/ vào Google Drive thành công!")

In [ ]:
# 5. Mở Streamlit Live Dashboard thông qua ngrok tunnel
from pyngrok import ngrok
import subprocess
import time

# Nhập Authtoken từ https://dashboard.ngrok.com (nếu có)
# ngrok.set_auth_token("YOUR_AUTHTOKEN_HERE")

proc = subprocess.Popen(["streamlit", "run", "src/dashboard/app.py", "--server.port", "8501"])
time.sleep(3)
public_url = ngrok.connect(8501)
print(f"🚀 VisionProf Dashboard đang mở tại: {public_url}")